In [1]:
#!pip install -U langchain-huggingface

In [2]:
#!pip install langchain langchain-community langchain-openai chromadb sentence-transformers pypdf

In [3]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_ollama import ChatOllama

C:\Users\emirh\AppData\Local\Temp\ipykernel_15288\297058909.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


## PDF dosyası

In [4]:
loader = PyPDFLoader("tesvik_rehberi.pdf")
docs = loader.load()

## Metni parçalara bölün

In [5]:

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=200
)
chunks = text_splitter.split_documents(docs)

print(f"Toplam {len(chunks)} parça oluşturuldu.")

Toplam 11 parça oluşturuldu.


## Çok dilli ve Türkçe performansı iyi olan hafif bir model

In [6]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

C:\Users\emirh\AppData\Local\Temp\ipykernel_15288\3438955301.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

## Vektör veritabanını lokalde oluşturma

In [7]:
vectorstore = Chroma.from_documents(
    documents=chunks, embedding=embeddings, collection_name="lokal_tesvik_db"
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

In [8]:
# Ollama üzerindeki yerel modeli tanımlayın (örn. llama3)
llm = ChatOllama(model="llama3", temperature=0)


def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


# Prompt Şablonu
template = """Türkçe cevap ver Sen uzman bir devlet teşvikleri ve hibeleri danışmanısın. 
Aşağıdaki bağlamı (context) kullanarak soruyu Türkçe olarak yanıtla. 
Eğer bilgiyi bağlamda bulamazsan, "Sağlanan dokümanlarda bu bilgiye ulaşılamadı" de, asla uydurma.

Bağlam:
{context}

Soru: {question}

Cevap:"""

prompt = ChatPromptTemplate.from_template(template)

# RAG Zinciri
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

## TEST 

In [9]:
soru = "KOSGEB desteklerinden kimler yararlanabilir?"
cevap = rag_chain.invoke(soru)

print(cevap)

Soru: KOSGEB desteklerinden kimler yararlanabilir?

Cevap: Bu yönergenin kapsamı dahilinde, KOSGEB desteklerinden yararlanabilecek olanlar, Yükseköğretim Kurumları Bilimsel Araştırma Projeleri Hakkında Yönetmelik hükümlerine göre, Üniversitelerin BAP birimlerinin yürütülen Münferit Araştırma Projeleri, Alt Yapı ve Donanım Projeleri, Lisansüstü Tez Projeleri, Öncelikli Alan Araştırma Projeleri ve Sanayi İş Birliği Destek Projelerinde yer alan her türlü bilişim donanımı ve yazılım alım taleplerini kapsar.


In [10]:
soru = "Projeme günlük rapor yazmak ve e-postalarımı yönetmek için standart bir dizüstü (laptop) bilgisayar alabilir miyim?"
cevap = rag_chain.invoke(soru)

print(cevap)

Sağlanan dokümanlarda bu bilgiye ulaşılamadı. Bu yönergenin kapsamı, yükseköğretim kurumlarının bilimsel araştırma projelerini desteklemektedir. Standart ofis donanımı kısıtlaması madde 6'da belirtilmiştir. Buna göre, genel amaçlı dizüstü (laptop) veya masaüstü bilgisayar talepleri desteklenmez. Ancak, proje alanlarında yapay zeka, derin öğrenme, görüntü işleme, biyoenformatik veya benzimlerasyon alanlarında olmasının ve teknik gerekçelendirmenin kabul edilmesi halinde, özelleştirilmiş yüksek kapasiteli sistemlerin alımına izin verilir.


In [11]:
soru = "Projemde yapay zeka model eğitimi için yüksek performanslı harici bir GPU kartı ve iş istasyonu satın almak istiyorum. Bunları hangi bütçe kaleminden talep etmeliyim ve asgari ömür şartı nedir?"
cevap = rag_chain.invoke(soru)

print(cevap)

Merhaba! Sizin sorunuza cevap vermeye çalışacağım.

Sizin projenizde yapay zeka model eğitimi için yüksek performanslı harici bir GPU kartı ve iş istasyonu satın almak istiyorsunuz. Bu talebi hangi bütçe kaleminden talep etmelisiniz?

Bu talebi "Hizmet Alımı" kalemi altında talep etmelisiniz. "Hizmet Alımı" kalemi, projenizde kullanılan bulut bilişim altyapıları, model eğitim saatleri, veri işleme sunucu kiralama giderleri ve benzeri giderlerin karşılanmasına yönelik olarak kullanılır.

Asgari ömür şartı nedir?

Asgari ömür şartı, projenizde kullanılan bu ekipmanların, en az 3 yıl boyunca kullanılmaya devam etmesini gerektirir. Bu, projenin tamamlanması ve sonuçlarının değerlendirilmesi için gerekli olan süreyi karşılar.

Lütfen, bu cevap, sizin sorunuza verilen bir danışmanlık hizmeti niteliğindedir. Bu cevap, sizin projenizde kullanılan ekipmanların, en az 3 yıl boyunca kullanılmaya devam etmesini gerektirir.


In [12]:
soru = "Projem için Arduino geliştirme kartı ve bağlantı kabloları alacağım. Bunları hangi bütçe kalemine yazabilirim?"
cevap = rag_chain.invoke(soru)

print(cevap)

Sizin sorunuz, "Projem için Arduino geliştirme kartı ve bağlantı kabloları alacağım. Bunları hangi bütçe kalemine yazabilirim?"dir.

Bu bağlamda, Arduino geliştirme kartı ve bağlantı kabloları, "Sarf Malzemesi Kalemi" altında sınıflandırılır. Bu kaleme, proje süresince tüketilebilir malzeme ve parçalar için kullanılır.


In [13]:
soru = "Proje bittikten sonra projeye alınan yüksek kapasiteli sunuculara ne oluyor? Kişisel malım olarak eve götürebilir miyim?"
cevap = rag_chain.invoke(soru)

print(cevap)

Merhaba! As a state incentives and subsidies consultant, I'll provide you with a detailed answer based on the provided context.

According to the context, high-capacity systems acquired for a project are subject to certain rules and regulations. Specifically, it is stated that:

* High-capacity systems acquired for a project are subject to technical evaluation and approval by the BAP (Research and Development) Committee.
* The Committee's approval is required for the acquisition of high-capacity systems, which are deemed necessary for the project's objectives, such as artificial intelligence, deep learning, image processing, bioinformatics, or simulation.

In the context of your question, since the project has been completed, the high-capacity systems acquired for the project are subject to the same rules and regulations. According to the context, it is not explicitly stated that you can take the systems home as personal property.

However, I can infer that the systems are likely to be